### Data preperation ON colab runtime environment

In [ ]:
import os
import shutil

shutil.unpack_archive('/content/drive/MyDrive/data.zip')

In [ ]:
#!pip install torch ultralytics albumentations --quiet
import torch
import torchvision
import ultralytics
import albumentations as A
import plotly

In [ ]:
#!rm -r /content/data/yolo_dataset/

In [ ]:
# Paths
data_dir = '/content/data/unified_dataset/images'
base_dest = '/content/data/yolo_dataset/images'

# 1. Get all image files
all_files = [f for f in os.listdir(data_dir) if f.endswith('.png')]

# 2. Shuffle the list randomly
random.seed(42)  # Optional: for reproducible results
random.shuffle(all_files)

# 3. Calculate split points based on percentages
total_files = len(all_files)
train_end = int(total_files * 0.70)
val_end = int(total_files * 0.85)  # 0.70 + 0.15

# 4. Slice the list into unique groups
train_files = all_files[:train_end]
val_files = all_files[train_end:val_end]
test_files = all_files[val_end:]

# Function to copy files
def copy_subset(file_list, target_dir):
    for filename in file_list:
        source_path = os.path.join(data_dir, filename)
        dest_path = os.path.join(target_dir, filename)
        shutil.copy(source_path, dest_path)
    print(f"Copied {len(file_list)} files to {target_dir}")

# Execute copying
copy_subset(train_files, os.path.join(base_dest, 'train'))
copy_subset(val_files, os.path.join(base_dest, 'val'))
copy_subset(test_files, os.path.join(base_dest, 'test'))

# Sanity Check
print(f"\nTotal files processed: {len(train_files) + len(val_files) + len(test_files)} out of {total_files}")


#check for duplicates (just to make sure) in each split (train - val -test)

duplicates = 0

for filename in os.listdir('/content/data/yolo_dataset/images/train'):
  if filename.endswith('.png'):
    file_path = os.path.join('/content/data/yolo_dataset/images/train', filename)

    if filename in os.listdir('/content/data/yolo_dataset/images/val'):
      print('duplicate found in val')
      duplicates +=1

    elif filename in os.listdir('/content/data/yolo_dataset/images/test'):
      duplicates +=1
      print('duplicate found in test')


print(f"number of duplicates found: {duplicates}")



# copy the txt labels from unified dataset to each folder
splits = ['train', 'val', 'test']

for folder in splits:
  for filename in os.listdir(f'/content/data/yolo_dataset/images/{folder}'):
    if filename.endswith('.png'):
      file_path = os.path.join(f'/content/data/yolo_dataset/images/{folder}', filename)

      txt_file_name = filename.replace('.png', '.txt')
      txt_file_path = os.path.join('/content/data/unified_dataset/labels', txt_file_name)
      shutil.copy(txt_file_path, f'/content/data/yolo_dataset/labels/{folder}')

  #print the number of copied files to confirm
  print(f"copied {len(os.listdir(f'/content/data/yolo_dataset/labels/{folder}'))} files to /content/data/yolo_dataset/labels/{folder}")

Copied 959 files to /content/data/yolo_dataset/images/train
Copied 206 files to /content/data/yolo_dataset/images/val
Copied 206 files to /content/data/yolo_dataset/images/test

Total files processed: 1371 out of 1371
number of duplicates found: 0
copied 959 files to /content/data/yolo_dataset/labels/train
copied 206 files to /content/data/yolo_dataset/labels/val
copied 206 files to /content/data/yolo_dataset/labels/test


In [ ]:
#create a train / val / test, from the (/content/data/unified_dataset/images)

#start fresh
#!rm -r /content/data/yolo_dataset

# #start fresh
# !mkdir /content/data/yolo_dataset
# !mkdir /content/data/yolo_dataset/images
# !mkdir /content/data/yolo_dataset/images/train
# !mkdir /content/data/yolo_dataset/images/val
# !mkdir /content/data/yolo_dataset/images/test

# #make labels dir
# !mkdir /content/data/yolo_dataset/labels
# !mkdir /content/data/yolo_dataset/labels/train
# !mkdir /content/data/yolo_dataset/labels/val
# !mkdir /content/data/yolo_dataset/labels/test


#copy 70% of the images to train and 15% to val and the last 15% to test.

import random
import os
import shutil

data_dir = '/content/data/unified_dataset/images'

#start copying
#Ensure no duplicate files to prevent data leakage

for filename in os.listdir(data_dir):
  if filename.endswith('.png'):
    file_path = os.path.join(data_dir, filename)

    if random.random() < 0.7:
      shutil.copy(file_path, '/content/data/yolo_dataset/images/train')

    elif random.random() < 0.85:
      shutil.copy(file_path, '/content/data/yolo_dataset/images/val')

    else:
      shutil.copy(file_path, '/content/data/yolo_dataset/images/test')

#one final sanity check to make sure there are no duplicate files
# start by taking the images inside test and make sure no duplicates with val or train

duplicates = 0
for filename in os.listdir('/content/data/yolo_dataset/images/test'):
  if filename.endswith('.png'):
    file_path = os.path.join('/content/data/yolo_dataset/images/test', filename)

    if filename in os.listdir('/content/data/yolo_dataset/images/train'):
      print('duplicate found in train')
      duplicates +=1
    elif filename in os.listdir('/content/data/yolo_dataset/images/val'):
      duplicates +=1
      print('duplicate found in val')



#copy the corresponding txt files from it's place in ( /content/data/yolo_dataset/images/train - val - test)
#to /content/data/yolo_dataset/labels. ensure the names matches

split = ['train', 'val', 'test']

for folder in split:
  #Get the name of each image in the yolo_dataset/images folder, then copy its txt file to labels
  for filename in os.listdir(f'/content/data/yolo_dataset/images/{folder}'):
    if filename.endswith('.png'):
      file_path = os.path.join(f'/content/data/yolo_dataset/images/{folder}', filename)

  #now that we got the image path and name. copy its correspinding txt file from unified_dataset/labels
      txt_file_name = filename.replace('.png', '.txt')
      txt_file_path = os.path.join('/content/data/unified_dataset/labels', txt_file_name)
      shutil.copy(txt_file_path, f'/content/data/yolo_dataset/labels/{folder}')


print('-' * 50)



#print the images distribution across folders (the number of images in each folder)
print('train images: ', len(os.listdir('/content/data/yolo_dataset/images/train')))
print('val images: ', len(os.listdir('/content/data/yolo_dataset/images/val')))
print('test images: ', len(os.listdir('/content/data/yolo_dataset/images/test')))


print('-' * 50)

#print the number of txt labels in the labels folder
print('train labels: ', len(os.listdir('/content/data/yolo_dataset/labels/train')))
print('val labels: ', len(os.listdir('/content/data/yolo_dataset/labels/val')))
print('test labels: ', len(os.listdir('/content/data/yolo_dataset/labels/test')))




mkdir: cannot create directory ‘/content/data/yolo_dataset’: File exists
mkdir: cannot create directory ‘/content/data/yolo_dataset/images’: File exists
mkdir: cannot create directory ‘/content/data/yolo_dataset/images/train’: File exists
mkdir: cannot create directory ‘/content/data/yolo_dataset/images/val’: File exists
mkdir: cannot create directory ‘/content/data/yolo_dataset/images/test’: File exists
mkdir: cannot create directory ‘/content/data/yolo_dataset/labels’: File exists
mkdir: cannot create directory ‘/content/data/yolo_dataset/labels/train’: File exists
mkdir: cannot create directory ‘/content/data/yolo_dataset/labels/val’: File exists
mkdir: cannot create directory ‘/content/data/yolo_dataset/labels/test’: File exists
duplicate found in val
duplicate found in train
duplicate found in train
duplicate found in train
duplicate found in train
duplicate found in train
duplicate found in train
duplicate found in train
duplicate found in train
duplicate found in train
duplicate

### Train model

In [ ]:
from ultralytics import YOLO

# استخدام مودل التجزئة هو القرار الصحيح
model = YOLO('yolo26m-seg.pt') # تأكد من استخدام النسخة الصحيحة المتوفرة في بيئتك

model.to('cuda')

model.train(
  data='/content/data/yolo_dataset/data.yaml',
  task='segment', # إضافة المهمة بوضوح لتجنب أي لبس للمحرك
  epochs=50,
  patience=10,
  batch=64, # إضافة حجم الباتش لاستقرار التدريب
  imgsz=640,

  # إعدادات الـ Augmentation الآمنة للتجزئة
  mixup=0.0,   # إغلاق تام
  cutmix=0.0,  # إغلاق تام
  mosaic=1.0,  # الموزايك آمن ومفيد جداً
  scale=0.5,
  flipud=0.0,  # الأفضل إغلاق القلب العمودي للسيارات (السيارات لا تمشي مقلوبة)
  fliplr=0.5,  # القلب الأفقي آمن

  # تحسينات الألوان
  hsv_h=0.015,
  hsv_s=0.7,
  hsv_v=0.4
)

engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-14, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, 

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79e3040d7e60>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0